[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.6_cache_aware_routing/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.6_cache_aware_routing/lab.ipynb)

# Lab 7.6: Cache-Aware Routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.6_cache_aware_routing/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.6_cache_aware_routing/lab.ipynb)

Simulate prefix-aware routing, session affinity savings, semantic cache
hit/miss tradeoffs, and cost impact at production scale.

In [ ]:
# Install dependencies using subprocess (works on Colab and Molab)
import subprocess, sys
# Install numpy for array math and matplotlib for plotting
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
# Import numpy for numerical operations
import numpy as np
# Import matplotlib for visualization
import matplotlib.pyplot as plt
# Import hashlib for prefix hashing (consistent routing)
import hashlib
# Import defaultdict to track per-node cache contents
from collections import defaultdict

# Set random seed for reproducible simulations
np.random.seed(42)
# Use clean plot style with grid
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Round-Robin vs Prefix-Aware Routing

Round-robin scatters requests without considering cache state.
Prefix-aware routing hashes the system prompt to a consistent node.

In [ ]:
# === SIMULATION PARAMETERS ===
# Total number of requests to simulate
NUM_REQUESTS = 1000
# Number of GPU serving nodes in the cluster
NUM_NODES = 8
# Number of distinct system prompt templates (e.g. different app contexts)
NUM_TEMPLATES = 5

# Create the template pool (these represent different system prompts)
TEMPLATES = [f'system_prompt_{i}' for i in range(NUM_TEMPLATES)]
# Generate request stream: each request has a random template prefix + unique query
requests = [(np.random.choice(TEMPLATES), f'user_query_{i}') for i in range(NUM_REQUESTS)]

def round_robin_hit_rate(requests, num_nodes):
    """Simulate round-robin routing and count prefix cache hits.
    Each node maintains its own independent cache of seen prefixes."""
    # Track which prefixes each node has already cached
    node_caches = defaultdict(set)
    # Counter for cache hits
    hits = 0
    for i, (prefix, _) in enumerate(requests):
        # Round-robin: assign request i to node (i % num_nodes)
        node = i % num_nodes
        # Check if this node already has this prefix cached
        if prefix in node_caches[node]:
            # Cache hit: KV blocks for this prefix can be reused
            hits += 1
        else:
            # Cache miss: node must compute full prefill for this prefix
            node_caches[node].add(prefix)
    # Return hit rate as fraction of total requests
    return hits / len(requests)

def prefix_aware_hit_rate(requests, num_nodes):
    """Simulate prefix-aware routing: consistent hash maps prefix to node.
    All requests sharing a prefix always go to the same node."""
    # Track cached prefixes per node
    node_caches = defaultdict(set)
    # Counter for cache hits
    hits = 0
    for prefix, _ in requests:
        # Hash the prefix string to get a consistent node assignment
        # MD5 provides uniform distribution across nodes
        node = int(hashlib.md5(prefix.encode()).hexdigest(), 16) % num_nodes
        # Check if the consistently-assigned node has this prefix cached
        if prefix in node_caches[node]:
            # Cache hit: prefix KV blocks already in GPU memory
            hits += 1
        else:
            # First request with this prefix to this node: cold miss
            node_caches[node].add(prefix)
    # Return hit rate
    return hits / len(requests)

# Run both routing strategies
rr_rate = round_robin_hit_rate(requests, NUM_NODES)
pa_rate = prefix_aware_hit_rate(requests, NUM_NODES)

# Visualize the comparison as a bar chart
fig_2, ax_2 = plt.subplots(figsize=(7, 4))
# Red for worse strategy, green for better
bars = ax_2.bar(['Round-Robin', 'Prefix-Aware'], [rr_rate, pa_rate],
              color=['#ef4444', '#22c55e'], edgecolor='black', width=0.5)
ax_2.set_ylabel('Cache Hit Rate')
ax_2.set_title('Prefix Cache Hit Rate by Routing Strategy')
ax_2.set_ylim(0, 1.05)
# Add text labels above each bar showing exact percentage
for bar, rate in zip(bars, [rr_rate, pa_rate]):
    ax_2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
# Print numeric results
print(f'Round-Robin: {rr_rate:.1%} hit rate')
print(f'Prefix-Aware: {pa_rate:.1%} hit rate')
# Compute relative improvement
print(f'Improvement: {(pa_rate - rr_rate) / max(rr_rate, 0.001) * 100:.0f}% higher hit rate')

## 2. Session Affinity: Multi-Turn Savings

Without affinity, every turn recomputes the full conversation history.
With affinity, only new tokens need prefill (prior KV cache is reused).

In [ ]:
# === PARAMETERS ===
# Number of independent chat sessions to simulate
NUM_CONVS = 50
# Number of turns in each conversation
TURNS = 10
# Tokens added per turn (user message + model response)
TOK_PER_TURN = 200

def simulate_prefill(use_affinity):
    """Compute cumulative prefill tokens across all conversations.
    With affinity: only new tokens per turn (cache hit on history).
    Without: full context recomputed each turn (landed on random node)."""
    # Track running total of prefill tokens
    cumulative = []
    total = 0
    for conv in range(NUM_CONVS):
        for turn in range(TURNS):
            # Total context at this turn = all prior turns + current
            context_len = (turn + 1) * TOK_PER_TURN
            if use_affinity:
                # Same node as last turn: only prefill the new tokens
                prefill = TOK_PER_TURN
            else:
                # Random node: must prefill entire conversation history
                prefill = context_len
            # Accumulate total prefill work
            total += prefill
            cumulative.append(total)
    return cumulative

# Run both scenarios
no_aff = simulate_prefill(use_affinity=False)
with_aff = simulate_prefill(use_affinity=True)

# Compute overall savings percentage
savings_pct = (1 - with_aff[-1] / no_aff[-1]) * 100

# Plot cumulative prefill tokens over time
fig_3, ax_3 = plt.subplots(figsize=(10, 5))
turns_axis = range(len(no_aff))
# Red line: no affinity (wasteful)
ax_3.plot(turns_axis, np.array(no_aff) / 1e6, label='No Affinity', color='#ef4444', linewidth=2)
# Green line: with affinity (efficient)
ax_3.plot(turns_axis, np.array(with_aff) / 1e6, label='Session Affinity', color='#22c55e', linewidth=2)
# Shade the area between curves to highlight savings
ax_3.fill_between(turns_axis, np.array(with_aff)/1e6, np.array(no_aff)/1e6,
                alpha=0.15, color='green')
ax_3.set_xlabel('Request Index (50 conversations x 10 turns)')
ax_3.set_ylabel('Cumulative Prefill Tokens (millions)')
ax_3.set_title(f'Session Affinity Saves {savings_pct:.0f}% of Prefill Compute')
ax_3.legend()
plt.tight_layout()
plt.show()
# Print summary statistics
print(f'Without affinity: {no_aff[-1]:,} total prefill tokens')
print(f'With affinity:    {with_aff[-1]:,} total prefill tokens')
print(f'Savings: {savings_pct:.1f}% fewer tokens to compute')

## 3. Semantic Cache: Threshold vs Hit Rate Tradeoff

Semantic caching matches queries by embedding similarity. Higher thresholds
reduce false positives but also reduce hit rate.

In [ ]:
# === PARAMETERS ===
# Embedding dimension (simulated with random vectors)
EMBED_DIM = 128
# Number of cached canonical query embeddings
CACHE_SIZE = 100
# Number of test queries to evaluate against cache
NUM_QUERIES = 500
# Noise level for simulated paraphrases (lower = closer to original)
NOISE = 0.1

# Build cache: 100 random unit vectors representing canonical queries
cache_embs = np.random.randn(CACHE_SIZE, EMBED_DIM)
# Normalize to unit length for cosine similarity via dot product
cache_embs /= np.linalg.norm(cache_embs, axis=1, keepdims=True)

def eval_threshold(threshold):
    """Evaluate semantic cache performance at given similarity threshold.
    First half of queries are paraphrases (should hit).
    Second half are novel queries (should miss).
    Returns (hit_rate, false_positive_rate)."""
    # Count total hits and false positives
    hits = 0
    fps = 0
    for i in range(NUM_QUERIES):
        # First half: paraphrases of cached queries
        is_paraphrase = i < NUM_QUERIES // 2
        if is_paraphrase:
            # Create paraphrase by adding small noise to cached embedding
            orig = cache_embs[i % CACHE_SIZE]
            query = orig + np.random.randn(EMBED_DIM) * NOISE
        else:
            # Novel query: completely random embedding
            query = np.random.randn(EMBED_DIM)
        # Normalize query for cosine similarity computation
        query /= np.linalg.norm(query)
        # Compute cosine similarity against all cached embeddings
        # (dot product of normalized vectors = cosine similarity)
        best_sim = (cache_embs @ query).max()
        # Check if best match exceeds threshold
        if best_sim >= threshold:
            hits += 1
            # False positive: novel query incorrectly matched to cache
            if not is_paraphrase:
                fps += 1
    # Hit rate: fraction of all queries that hit the cache
    hit_rate = hits / NUM_QUERIES
    # FP rate: fraction of novel queries that incorrectly match
    fp_rate = fps / (NUM_QUERIES // 2)
    return hit_rate, fp_rate

# Sweep thresholds from permissive (0.50) to strict (0.98)
thresholds = np.arange(0.50, 0.99, 0.04)
# Evaluate each threshold
results = [eval_threshold(t) for t in thresholds]
# Extract hit rates and false positive rates
hit_rates = [r[0] for r in results]
fp_rates = [r[1] for r in results]

# Create dual-axis plot showing both metrics against threshold
fig_4, ax1 = plt.subplots(figsize=(9, 5))
# Second y-axis for false positive rate
ax2 = ax1.twinx()
# Blue line: cache hit rate (higher is better)
l1 = ax1.plot(thresholds, hit_rates, 'o-', color='#2563eb', linewidth=2, label='Hit Rate')
# Red line: false positive rate (lower is better)
l2 = ax2.plot(thresholds, fp_rates, 's--', color='#dc2626', linewidth=2, label='False Positive Rate')
ax1.set_xlabel('Cosine Similarity Threshold')
ax1.set_ylabel('Cache Hit Rate', color='#2563eb')
ax2.set_ylabel('False Positive Rate', color='#dc2626')
ax1.set_title('Semantic Cache: Hit Rate vs False Positive Tradeoff')
ax1.set_ylim(0, 1)
ax2.set_ylim(0, 1)
# Combined legend from both axes_4
lines = l1 + l2
ax1.legend(lines, [l.get_label() for l in lines], loc='center right')
plt.tight_layout()
plt.show()
# Find recommended threshold where FP drops below 5%
safe_idx = next((i for i, fp in enumerate(fp_rates) if fp < 0.05), len(fp_rates)-1)
print(f'Recommended threshold: {thresholds[safe_idx]:.2f}')
print(f'  Hit rate: {hit_rates[safe_idx]:.1%}, FP rate: {fp_rates[safe_idx]:.1%}')

## 4. Cost Impact Analysis

Estimate monthly savings from prefix + semantic caching at different scales.

In [ ]:
# === COST MODEL PARAMETERS ===
# Cost per 1K input tokens (typical managed API pricing)
COST_PER_1K = 0.01
# Average prefix length that can be cached (system prompt)
AVG_PREFIX = 500
# Prefix cache hit rate achievable with prefix-aware routing
PREFIX_HR = 0.80
# Semantic cache hit rate (full response returned without LLM call)
SEMANTIC_HR = 0.30
# Average total tokens per request (prefix + user query + output)
AVG_TOTAL = 1500

# Traffic levels to analyze
traffic = [1_000, 10_000, 100_000]  # requests per day
labels = ['1K/day', '10K/day', '100K/day']

# Compute savings for each traffic level
baselines, prefix_saves, semantic_saves = [], [], []
for daily in traffic:
    # Scale to monthly request count
    monthly = daily * 30
    # Baseline monthly cost: all requests at full price
    baseline = monthly * AVG_TOTAL / 1000 * COST_PER_1K
    # Prefix savings: avoid prefilling cached tokens (80% of the time)
    p_save = monthly * AVG_PREFIX / 1000 * COST_PER_1K * PREFIX_HR
    # Semantic savings: avoid entire LLM call for semantic hits
    s_save = monthly * SEMANTIC_HR * AVG_TOTAL / 1000 * COST_PER_1K
    baselines.append(baseline)
    prefix_saves.append(p_save)
    semantic_saves.append(s_save)

# Create grouped bar chart showing savings breakdown
fig_5, ax_5 = plt.subplots(figsize=(9, 5))
x = np.arange(len(labels))
w = 0.25  # bar width
# Green: prefix cache savings
ax_5.bar(x - w, prefix_saves, w, label='Prefix Cache', color='#22c55e', edgecolor='black')
# Blue: semantic cache savings
ax_5.bar(x, semantic_saves, w, label='Semantic Cache', color='#3b82f6', edgecolor='black')
# Amber: total combined savings
total = [p + s for p, s in zip(prefix_saves, semantic_saves)]
ax_5.bar(x + w, total, w, label='Combined', color='#f59e0b', edgecolor='black')
ax_5.set_xticks(x)
ax_5.set_xticklabels(labels)
ax_5.set_ylabel('Monthly Savings ($)')
ax_5.set_title('Monthly Cost Savings from Cache-Aware Routing')
ax_5.legend()
plt.tight_layout()
plt.show()

# Print summary table
for label, base, ps, ss in zip(labels, baselines, prefix_saves, semantic_saves):
    # Show baseline cost and savings breakdown for each traffic level
    print(f'{label}: baseline ${base:,.0f}/mo | saves ${ps+ss:,.0f} ({(ps+ss)/base*100:.0f}%)')

## Key Takeaways

1. Prefix-aware routing achieves near-perfect hit rates vs ~75-80% with round-robin
2. Session affinity saves 60-80% of prefill compute in multi-turn conversations
3. Semantic cache threshold ~0.90 balances hit rate vs false positive correctness
4. Combined caching saves thousands per month at 100K+ req/day scale
5. Staleness detection and invalidation are critical for production correctness